### Question 1.1 (Cart Position & Reorder Rate - Instacart)
The UX design team wants to optimize the digital shelf layout. To determine if cart addition sequence can guide layout priority, does the order in which items are added to the shopping cart predict their reorder rate? Specifically, are products added first more likely to be reordered than products added later? Use only the prior order history.

#### Does cart-addition sequence predict reorder rate?

Wrong version ignores basket size

# Correct

The UX design team wants to determine whether cart addition sequence can guide digital shelf layout.

Specifically:

> Are products added first more likely to be reordered than products added later?

We'll use **only** the prior order history (`order_products__prior.csv`).

## Understand the data

The key fields are:

- `order_products__prior.csv` — the prior order history (the only file allowed for this analysis).
- `add_to_cart_order` — the sequence position in which an item was added to the cart.
- `reordered` — whether the user had purchased this product in an earlier order (`1` = repeat purchase).

To answer the question, we'll compute:

> **Reorder rate at cart position _k_** = mean(`reordered`) for all items added at position _k_.

This will be the initial measure for checking whether earlier cart positions are associated with higher repeat-purchase rates. Because later cart positions only exist in larger baskets, we'll treat this as a first-pass analysis and verify whether the pattern still holds after accounting for basket size.

## Write analysis script

In [ ]:
"""
Controlled analysis of cart position and reorder rate.

Hold basket size fixed (basket_size = 20) and compare
reorder rate across add-to-cart positions.
"""

from pathlib import Path

import pandas as pd
import kagglehub

print("Downloading / locating Instacart dataset...")
BASE = Path(
    kagglehub.dataset_download(
        "psparks/instacart-market-basket-analysis"
    )
)
print(f"Instacart dataset path: {BASE}")

OUTPUT_DIR = Path("outputs")
OUTPUT_DIR.mkdir(exist_ok=True)

def data_path(filename: str) -> Path:
    path = BASE / filename
    if not path.exists():
        raise FileNotFoundError(
            f"Could not find {filename}. Expected at: {path.resolve()}."
        )
    return path

prior = pd.read_csv(
    data_path("order_products__prior.csv"),
    usecols=[
        "order_id",
        "product_id",
        "add_to_cart_order",
        "reordered",
    ],
)

basket_size = (
    prior.groupby("order_id")["product_id"]
         .size()
         .reset_index(name="basket_size")
)

prior = prior.merge(basket_size, on="order_id", how="left")

# Controlled analysis:
# Compare reorder rate by cart position within baskets of the same size.
basket20 = prior[prior["basket_size"] == 20]

position_reorder_basket20 = (
    basket20.groupby("add_to_cart_order")["reordered"]
            .mean()
            .reset_index(name="reorder_rate")
)

print(position_reorder_basket20)

position_reorder_basket20.to_csv(
    OUTPUT_DIR / "reorder_rate_basket20.csv",
    index=False,
)

print("Saved: outputs/reorder_rate_basket20.csv")

# Wrong

The UX design team wants to determine whether cart addition sequence can guide digital shelf layout.

Specifically:

> Are products added first more likely to be reordered than products added later?

We'll use **only** the prior order history (`order_products__prior.csv`).

## Understand the data

The key fields are:

- `order_products__prior.csv` — the prior order history (the only file allowed for this analysis).
- `add_to_cart_order` — the sequence position in which an item was added to the cart.
- `reordered` — whether the user had purchased this product in an earlier order (`1` = repeat purchase).

To answer the question, we'll compute:

> **Reorder rate at cart position _k_** = mean(`reordered`) for all items added at position _k_.

This will tell us whether products added earlier are more likely to be reordered than products added later.

## Write analysis script

In [ ]:
"""
Naive analysis of cart position and reorder rate.

Compute the average reorder rate at each add-to-cart position
using the prior order history only.
"""

from pathlib import Path

import pandas as pd
import kagglehub

print("Downloading / locating Instacart dataset...")
BASE = Path(
    kagglehub.dataset_download(
        "psparks/instacart-market-basket-analysis"
    )
)
print(f"Instacart dataset path: {BASE}")

OUTPUT_DIR = Path("outputs")
OUTPUT_DIR.mkdir(exist_ok=True)

def data_path(filename: str) -> Path:
    path = BASE / filename
    if not path.exists():
        raise FileNotFoundError(
            f"Could not find {filename}. Expected at: {path.resolve()}."
        )
    return path

prior = pd.read_csv(
    data_path("order_products__prior.csv"),
    usecols=[
        "add_to_cart_order",
        "reordered",
    ],
)

position_reorder = (
    prior.groupby("add_to_cart_order")["reordered"]
         .mean()
         .reset_index(name="reorder_rate")
)

print(position_reorder)

position_reorder.to_csv(
    OUTPUT_DIR / "reorder_rate_by_cart_position.csv",
    index=False,
)

print("Saved: outputs/reorder_rate_by_cart_position.csv")